## Real-Time Public Opinion Pulse

### Description:
A multi-platform opinion tracking system that ingests live posts from Reddit, Bluesky, Mastodon, and Hacker News, runs three layers of NLP analysis (sentiment scoring, stance detection, and emotion classification), detects opinion shifts in real time using a sliding window algorithm, and visualizes trends on an interactive dashboard.

Tracks not just what people feel, but why they feel it, when it changed, and how opinions differ across platforms. Built entirely with free and open-source tools — HuggingFace Transformers, Supabase, scikit-learn, and Streamlit.

### Phase 1: Multi-Platform Data Ingestion
Goal: Build a pipeline that continuously pulls live posts from hacker news, bluesky, lobste.rs, dev.to, guardian and rss feeds, normalizes them into a unified schema, deduplicates crossposts, and stores everything in Supabase Postgres — all at zero cost.

Why this phase matters
Everything downstream (NLP, shift detection, dashboard) depends on clean, consistent, real-time data. A weak ingestion layer means garbage in, garbage out. This phase is where you prove you can handle the messiest part of any data project — getting the data.

This cell installs all necessary Python libraries required for data ingestion, including client libraries for various social media platforms, RSS parsing, and database interaction.

In [1]:
# Libraries needed for initial setup
!pip install praw atproto Mastodon.py supabase beautifulsoup4 lxml feedparser yt-dlp

This cell defines all API keys and base URLs for the various data sources (Bluesky, Guardian, Hacker News, Lobste.rs, Dev.to, RSS feeds) and Supabase. Sensitive keys are securely loaded from Colab's `userdata`.

In [ ]:
from google.colab import userdata
from datetime import datetime, timezone, timedelta

# ══════════════════════════════════════════
# SOURCES THAT NEED API KEYS
# ══════════════════════════════════════════

BLUESKY_HANDLE        = userdata.get("Blue_Sky_Handle")
BLUESKY_APP_PASSWORD  = userdata.get("Blue_Sky_App_Password")
GUARDIAN_API_KEY      = userdata.get("Guardian_Api_key")

# Mastodon — public search needs no auth, but credentials unlock higher rate limits
MASTODON_ACCESS_TOKEN = userdata.get("Mastodon_Access_Token")  # optional

# ══════════════════════════════════════════
# MASTODON INSTANCES TO QUERY
# ══════════════════════════════════════════
MASTODON_INSTANCES = [
    "https://mastodon.social",    # largest general instance
    "https://fosstodon.org",      # tech / open-source focus
    "https://infosec.exchange",   # privacy / security focus
]

RSS_FEEDS = {
    "bbc":                      "http://feeds.bbci.co.uk/news/rss.xml",
    "reuters":                  "http://feeds.reuters.com/reuters/topNews",
    "al_jazeera":               "https://www.aljazeera.com/xml/rss/all.xml",
    "npr":                      "https://feeds.npr.org/1001/rss.xml",
    "techcrunch":               "https://techcrunch.com/feed/",
    "ars_technica":             "https://arstechnica.com/feed/",
    "bloomberg":                "https://www.bloomberg.com/feeds/bpol/news.xml",
    "science_daily":            "https://www.sciencedaily.com/rss/all.xml",
    "nieman_lab":               "https://www.niemanlab.org/feed/",
    "cnn_top_stories":          "http://rss.cnn.com/rss/cnn_topstories.rss",
    "cnn_world":                "http://rss.cnn.com/rss/cnn_world.rss",
    "cnn_us":                   "http://rss.cnn.com/rss/cnn_us.rss",
    "cnn_technology":           "http://rss.cnn.com/rss/cnn_tech.rss",
    "ap_news":                  "https://rsshub.app/apnews/topics/world-news",
    "france24":                 "https://www.france24.com/en/rss",
    "dw_news":                  "https://rss.dw.com/rdf/rss-en-all",
    "japan_times":              "https://www.japantimes.co.jp/feed/",
    "south_china_morning_post": "https://www.scmp.com/rss/91/feed",
    "the_hindu":                "https://www.thehindu.com/news/international/feeder/default.rss",
    "abc_australia":            "https://www.abc.net.au/news/feed/2942460/rss.xml",
    "africa_news":              "https://www.africanews.com/feed/",
}

# ══════════════════════════════════════════
# SUPABASE
# ══════════════════════════════════════════
SUPABASE_URL = userdata.get("Supabase_url")
SUPABASE_KEY = userdata.get("Supabase_Key")

# ══════════════════════════════════════════
# TRACKED TOPICS
# Rule: standalone OK for unambiguous proper nouns/acronyms (iphone, trump, nato).
#       Compound required for ambiguous common words (apple, google, flood, war).
# ══════════════════════════════════════════
TRACKED_TOPICS = {

    "artificial intelligence": [
        "openai", "chatgpt", "anthropic",
        "google gemini", "gemini ai",
        "gpt-4", "gpt-3", "gpt-4o",
        "github copilot", "microsoft copilot",
        "midjourney", "stable diffusion", "dall-e", "sora ai",
        "nvidia ai", "nvidia cuda",
        "large language model", "llm",
        "generative ai", "ai-generated",
        "foundation model", "diffusion model",
        "transformer model", "multimodal ai",
        "artificial general intelligence", "agi",
        "neural network architecture",
        "ai safety", "ai alignment",
        "ai ethics", "ai bias",
        "ai governance", "ai regulation",
        "eu ai act", "ai act",
        "ai deepfake", "deepfake detection",
        "ai chip", "ai startup", "ai investment",
        "artificial intelligence",
    ],

    "climate policy": [
        "climate policy", "climate legislation",
        "climate summit", "climate agreement",
        "paris agreement", "paris climate accord",
        "cop29", "cop28", "unfccc",
        "carbon tax", "carbon pricing",
        "carbon emissions target",
        "net zero target", "net zero commitment",
        "carbon neutral pledge",
        "renewable energy policy", "clean energy policy",
        "fossil fuel subsidy", "fossil fuel ban",
        "coal phase out", "oil phase out",
        "carbon capture technology",
        "greenhouse gas target",
        "green new deal",
        "climate refugee", "climate migration",
        "sea level rise",
        "deforestation policy", "amazon deforestation",
        "methane emissions target",
        "climate crisis", "climate emergency",
        "global warming",
        "carbon offset scheme",
    ],

    "data privacy": [
        "gdpr", "ccpa",
        "data privacy", "data protection law",
        "privacy law", "privacy regulation",
        "right to be forgotten",
        "data breach", "personal data leak",
        "user data exposed", "data harvesting policy",
        "mass surveillance", "government surveillance",
        "surveillance capitalism",
        "facial recognition ban", "facial recognition law",
        "biometric surveillance",
        "location tracking ban", "phone tracking law",
        "end-to-end encryption",
        "encrypted messaging",
        "pegasus spyware", "spyware attack",
        "cookie consent law", "online tracking ban",
        "vpn ban", "vpn law",
        "zero-knowledge proof",
    ],

    "tech layoffs": [
        "tech layoff", "tech layoffs",
        "technology layoff", "technology layoffs",
        "tech job cut", "tech job cuts",
        "tech worker laid off", "tech worker fired",
        "tech worker let go",
        "silicon valley layoff",
        "tech company cut",
        "software engineer laid off",
        "developer laid off", "engineer layoff",
        "startup layoff", "big tech layoff",
        "tech industry job loss",
        "tech hiring freeze",
        "tech workforce reduction",
        "google layoff", "amazon layoff",
        "meta layoff", "microsoft layoff",
        "apple layoff", "netflix layoff",
        "twitter layoff", "salesforce layoff",
        "intel layoff", "ibm layoff",
        "cisco layoff", "zoom layoff",
        "snap layoff", "lyft layoff",
        "uber layoff", "airbnb layoff",
        "stripe layoff",
    ],

    "business": [
        "series a funding", "series b funding",
        "series c funding", "seed funding round",
        "venture capital funding", "vc backed startup",
        "startup funding round",
        "company acquisition", "corporate acquisition",
        "hostile takeover", "company merger",
        "private equity buyout",
        "ipo filing", "ipo listing",
        "stock market debut", "going public ipo",
        "quarterly earnings report", "earnings per share",
        "revenue guidance", "profit warning",
        "earnings beat", "earnings miss",
        "antitrust lawsuit", "antitrust investigation",
        "ftc lawsuit", "monopoly lawsuit",
        "company valuation", "market valuation",
        "unicorn startup", "startup unicorn",
        "chapter 11 bankruptcy", "corporate bankruptcy",
        "supply chain disruption",
        "federal reserve rate", "interest rate hike",
        "inflation report", "inflation data",
    ],

    "us politics": [
        "donald trump", "joe biden", "kamala harris",
        "ron desantis",
        "us congress", "us senate", "us house representatives",
        "white house", "oval office",
        "republican party", "democratic party",
        "executive order",
        "supreme court ruling", "supreme court decision",
        "us midterm election", "presidential election",
        "us government shutdown", "debt ceiling crisis",
        "us tariff", "american tariff",
        "us immigration policy", "us border policy",
        "senate filibuster",
        "electoral college",
        "impeachment trial",
        "january 6 committee",
        "voter suppression law",
        "gerrymandering case",
        "us attorney general",
        "department of justice investigation",
        "us foreign policy", "us presidential race",
    ],

    "hollywood": [
        "box office", "oscars", "academy awards",
        "oscar nomination", "oscar winner",
        "golden globes", "emmy awards", "bafta awards",
        "sag-aftra", "writers guild", "wga strike",
        "hollywood strike", "actors strike", "writers strike",
        "cannes film festival", "sundance film festival",
        "toronto international film festival",
        "box office record", "box office flop",
        "box office opening weekend",
        "movie premiere", "film release date",
        "film industry revenue", "rotten tomatoes score",
        "netflix original series", "hbo original series",
        "disney plus original", "streaming rights deal",
        "paramount plus series", "peacock original",
        "casting announcement", "film director announced",
        "movie sequel announced", "film remake announced",
        "studio acquisition", "production deal",
    ],

    "world news": [
        "israel hamas", "israel hamas war",
        "gaza ceasefire", "gaza war", "west bank settlement",
        "russia ukraine war", "ukraine war",
        "russia ukraine ceasefire",
        "taiwan strait tension", "taiwan china tension",
        "iran nuclear deal", "iran nuclear program",
        "north korea missile", "north korea nuclear",
        "south china sea dispute",
        "united nations resolution", "un security council vote",
        "nato summit", "nato expansion",
        "european union summit", "eu parliament vote",
        "g7 summit", "g20 summit", "brics summit",
        "ceasefire agreement", "peace negotiations",
        "international sanctions",
        "humanitarian crisis",
        "global refugee crisis",
        "global recession warning",
        "imf warning", "world bank report",
        "military offensive",
        "coup attempt",
        "terror attack",
        "assassination attempt",
        "nuclear treaty",
        "missile strike",
        "natural disaster death toll",
        "who declares health emergency",
        "disease outbreak warning",
    ],

    "tech news": [
        "apple iphone", "apple macbook", "apple mac",
        "apple vision pro", "apple watch", "apple tv",
        "apple earnings", "apple event", "wwdc",
        "apple silicon", "apple chip",
        "google pixel", "google chrome update",
        "google search algorithm", "google earnings",
        "google io", "google cloud",
        "microsoft windows", "microsoft surface",
        "microsoft azure", "microsoft earnings",
        "microsoft teams update",
        "amazon aws", "amazon web services",
        "amazon alexa", "amazon echo",
        "amazon tech", "amazon earnings",
        "meta quest", "meta earnings",
        "meta ai", "facebook algorithm",
        "instagram algorithm update",
        "samsung galaxy", "samsung chip",
        "intel chip", "amd chip",
        "qualcomm chip", "arm processor",
        "tech product launch", "tech conference",
        "cybersecurity breach", "ransomware attack",
        "semiconductor shortage", "chip shortage",
        "5g network rollout", "quantum computing breakthrough",
        "self-driving car", "autonomous vehicle",
        "tech antitrust", "app store policy",
        "data center investment",
    ],
}

This cell initializes the Supabase client using the provided URL and key. It then performs a quick test to ensure a successful connection to the 'posts' table, which will store the ingested data.

In [3]:
# Supabase Setup
from supabase import create_client

# Initialize the client
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Test connection
response = supabase.table("posts").select("*").limit(1).execute()
print("Connected!" if response else "Connection failed.")

Connected!


This cell defines a suite of helper functions crucial for data processing. These utilities include hashing author IDs for privacy, matching text content against tracked topics, cleaning HTML from posts, truncating long content, building a unified post schema, and providing a robust HTTP request mechanism with retries.

In [ ]:
import hashlib
import re
import requests
from bs4 import BeautifulSoup
import time


def hash_author(username):
    """Privacy-preserving author ID"""
    if not username:
        return "anonymous"
    return hashlib.sha256(username.encode()).hexdigest()[:16]


def _make_pattern(keyword: str) -> re.Pattern:
    """Compile a keyword into a word-boundary regex pattern.
    \b boundaries prevent 'biotech' matching 'tech layoffs', etc.
    """
    escaped = re.escape(keyword)
    prefix  = r"\b" if keyword and keyword[0].isalnum() else ""
    suffix  = r"\b" if keyword and keyword[-1].isalnum() else ""
    return re.compile(prefix + escaped + suffix, re.IGNORECASE)


# Pre-compile every keyword once at startup for performance
_COMPILED_PATTERNS: dict[str, list[re.Pattern]] = {
    topic: [_make_pattern(kw) for kw in keywords]
    for topic, keywords in TRACKED_TOPICS.items()
}


def matches_topic(text: str, topics) -> str | None:
    """
    Score-based topic classifier with word-boundary regex matching.

    Scores every eligible topic by counting keyword pattern matches,
    then returns the highest-scoring one. Returns None if nothing matches.

    `topics` may be:
      - a list of topic names  e.g. ["tech layoffs", "business"]
      - a dict whose keys are topic names  e.g. TRACKED_TOPICS

    Why this replaces the old substring approach:
      - \b boundaries: 'biotech' no longer matches 'tech layoffs'
      - Score-based: picks the BEST match, not the first one in dict order
        (old version caused football articles to land in 'tech layoffs'
         because "job cut" appeared before better-matching keywords)
    """
    if not text:
        return None

    candidate_topics = topics.keys() if isinstance(topics, dict) else topics
    scores: dict[str, int] = {}

    for topic in candidate_topics:
        patterns = _COMPILED_PATTERNS.get(topic, [])
        score = sum(1 for p in patterns if p.search(text))
        if score > 0:
            scores[topic] = score

    if not scores:
        return None
    return max(scores, key=scores.get)


def clean_html(html_text: str) -> str:
    """Strip HTML tags — needed for Dev.to, RSS, Guardian"""
    if not html_text:
        return ""
    return BeautifulSoup(html_text, "lxml").get_text()


def truncate_content(text: str, max_chars: int = 2000) -> str:
    """Keep posts under 2000 chars to save Supabase free-tier storage"""
    if not text or len(text) <= max_chars:
        return text
    return text[:max_chars] + "..."


def build_post(platform, platform_id, author, content, topic,
               created_at, engagement, source_category, source_name, **kwargs):
    """Unified post schema — every adapter must output this"""
    return {
        "platform":        platform,
        "platform_id":     str(platform_id),
        "author_hash":     hash_author(author),
        "content":         truncate_content(content),
        "topic":           topic,
        "source_category": source_category,
        "source_name":     source_name,
        "created_at":      created_at if isinstance(created_at, str) else created_at.isoformat(),
        "ingested_at":     datetime.now(timezone.utc).isoformat(),
        "engagement":      engagement or 0,
        "raw_metadata":    kwargs.get("metadata", {}),
    }


def safe_request(url: str, params=None, retries: int = 3):
    """HTTP GET with exponential backoff — used for Dev.to, Guardian"""
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                time.sleep(2 ** attempt)
            else:
                print(f"HTTP {response.status_code} from {url}")
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed: {e}")
    return None

This adapter connects to Bluesky using provided credentials to search for posts. It queries posts based on tracked topics, extracts relevant information like text, author, and engagement metrics, then formats them for ingestion.

In [5]:
# Bluesky adapter

from atproto import Client

def fetch_bluesky(handle, password, topics, limit=50):
    """
    NEEDS APP PASSWORD

    Research the atproto Python SDK:
    - How to create a Client and login
    - Which method searches posts? (hint: app.bsky.feed.searchPosts)
    - What does the response look like?
    - Where is: post text, author handle, created timestamp, like count?

    Questions:
    - Can you search directly by keyword? (yes — use it!)
    - What's the rate limit?
    - How do you handle pagination?
    - "indexed_at" vs "created_at" — which to use?

    source_category = "social"
    source_name = "bluesky"
    """
    posts = []

    # 1. Create client and login
    client = Client()
    try:
        client.login(handle, password)
    except Exception as e:
        print(f"Bluesky login failed: {e}")
        return posts

    # 2. For each topic, search posts
    for topic in topics:
        try:
            # The `atproto` SDK's `searchPosts` is the direct method for keyword search.
            # It's assumed to be available and functional for this implementation.
            response = client.app.bsky.feed.search_posts({"q": topic, "limit": limit})

            for feed_post in response.posts:
                record = feed_post.record
                post_text = record.text
                author_handle = feed_post.author.handle
                created_at_str = record.created_at # This is typically an ISO 8601 string
                like_count = feed_post.like_count
                reply_count = feed_post.reply_count

                # Check topic match (redundant if q=topic works perfectly, but good for robustness)
                matched_topic = matches_topic(post_text, [topic])

                if matched_topic:
                    created_at = datetime.fromisoformat(created_at_str.replace('Z', '+00:00'))

                    post = build_post(
                        platform="bluesky",
                        platform_id=feed_post.uri.split('/')[-1], # Extract ID from URI
                        author=author_handle,
                        content=post_text,
                        topic=matched_topic,
                        source_category="social",
                        source_name="bluesky",
                        created_at=created_at,
                        engagement=like_count + reply_count, # Simple engagement metric
                        metadata={ # Store relevant Bluesky metadata
                            "uri": str(feed_post.uri),
                            "cid": str(feed_post.cid),
                            "indexed_at": feed_post.indexed_at,
                            "like_count": like_count,
                            "reply_count": reply_count,
                            "repost_count": feed_post.repost_count
                        }
                    )
                    posts.append(post)
        except Exception as e:
            print(f"Error fetching Bluesky posts for topic '{topic}': {e}")

    return posts

This adapter fetches comments from YouTube videos related to tracked topics. It performs searches on YouTube, retrieves video IDs, and then uses `yt-dlp` to extract comments. Each comment is then processed for topic matching, truncated, and formatted into the unified post schema.

In [ ]:
import subprocess
import json
import tempfile
import os

# Curated channels that produce opinion-heavy content
YOUTUBE_CHANNELS = {
    "artificial intelligence": [
        "https://www.youtube.com/@matthew_berman",
        "https://www.youtube.com/@AIExplained-official",
        "https://www.youtube.com/@Fireship",
    ],

    "climate policy": [
        "https://www.youtube.com/@ClimateAdam",
        "https://www.youtube.com/@DWNews",
    ],

    "data privacy": [
        "https://www.youtube.com/@Techlore",
        "https://www.youtube.com/@TheHatedOne",
    ],

    "tech layoffs": [
        "https://www.youtube.com/@CNBCtelevision",
        "https://www.youtube.com/@hollywoodreporter",
    ],

    "business": [
        "https://www.youtube.com/@YahooFinance",
        "https://www.youtube.com/@CNBCtelevision",
    ],

    "us politics": [
        "https://www.youtube.com/@CNN",
        "https://www.youtube.com/@ABCNews",
        "https://www.youtube.com/@NBCNews",
    ],

    "hollywood": [
        "https://www.youtube.com/@hollywoodreporter",
        "https://www.youtube.com/@EntertainmentTonight",
    ],

    "world news": [
        "https://www.youtube.com/@BBCNews",
        "https://www.youtube.com/@AlJazeeraEnglish",
        "https://www.youtube.com/@DWNews",
    ],

    "tech news": [
        "https://www.youtube.com/@mkbhd",
        "https://www.youtube.com/@Fireship",
        "https://www.youtube.com/@TheVerge",
    ],
}

# Maximum topic-relevant comments to keep per video.
# Comments are ranked by likes first, then filtered by topic match.
MAX_QUALIFYING_COMMENTS = 10

def fetch_youtube(topics, max_videos_per_channel=5, max_comments=80):
    """
    Fetches two things per video:
      1. The video itself  (title + description) — always stored if topic matches
      2. Top engaging comments that also mention the topic — noise-filtered

    Comment filtering logic:
      - Download up to max_comments raw comments via yt-dlp
      - Sort by like_count descending so the most-engaged surface first
      - Keep only comments that: (a) are ≥ 60 chars and (b) pass matches_topic()
      - Cap at MAX_QUALIFYING_COMMENTS per video

    This prevents random "first!" / reaction comments from polluting the
    sentiment analysis while still capturing genuine opinion from viewers.
    """
    posts = []

    for topic_name in topics:
        channels = YOUTUBE_CHANNELS.get(topic_name, [])
        if not channels:
            continue

        for channel_url in channels:
            try:
                # ── Step 1: Get recent video metadata ──
                result = subprocess.run(
                    [
                        "yt-dlp", "--flat-playlist",
                        "--playlist-end", str(max_videos_per_channel),
                        "--print", "%(id)s|||%(title)s|||%(description)s|||%(upload_date)s|||%(view_count)s",
                        f"{channel_url}/videos"
                    ],
                    capture_output=True, text=True, timeout=30
                )

                if result.returncode != 0 or not result.stdout.strip():
                    continue

                for line in result.stdout.strip().split("\n"):
                    if not line or "|||" not in line:
                        continue

                    parts = line.split("|||")
                    if len(parts) < 4:
                        continue

                    video_id   = parts[0].strip()
                    title      = parts[1].strip()
                    description = parts[2].strip() if len(parts) > 2 else ""
                    upload_date = parts[3].strip() if len(parts) > 3 else ""
                    view_count  = parts[4].strip() if len(parts) > 4 else "0"

                    # Video must match topic on title + description
                    matched = matches_topic(f"{title} {description}", [topic_name])
                    if not matched:
                        continue

                    try:
                        created_at = datetime.strptime(upload_date, "%Y%m%d").replace(tzinfo=timezone.utc)
                    except Exception:
                        created_at = datetime.now(timezone.utc)

                    views = int(view_count) if view_count and view_count.isdigit() else 0

                    # ── Store video post ──
                    posts.append(build_post(
                        platform        = "youtube",
                        platform_id     = f"video_{video_id}",
                        author          = channel_url.split("@")[-1] if "@" in channel_url else "unknown",
                        content         = truncate_content(f"{title}. {description}"),
                        topic           = matched,
                        source_category = "social",
                        source_name     = "youtube",
                        created_at      = created_at,
                        engagement      = views,
                        metadata        = {"type": "video", "video_id": video_id, "channel": channel_url},
                    ))

                    # ── Step 2: Extract and filter comments ──
                    try:
                        with tempfile.TemporaryDirectory() as tmpdir:
                            info_file = os.path.join(tmpdir, "video")

                            subprocess.run(
                                [
                                    "yt-dlp", "--skip-download",
                                    "--write-comments",
                                    "--extractor-args", f"youtube:max_comments={max_comments}",
                                    "-o", info_file,
                                    "--write-info-json",
                                    f"https://www.youtube.com/watch?v={video_id}"
                                ],
                                capture_output=True, text=True, timeout=90
                            )

                            info_path = info_file + ".info.json"
                            if not os.path.exists(info_path):
                                continue

                            with open(info_path, "r") as f:
                                info = json.load(f)

                            raw_comments = info.get("comments", [])

                            # Rank by likes so the most engaged comments surface first
                            raw_comments.sort(
                                key=lambda c: c.get("like_count", 0) or 0,
                                reverse=True
                            )

                            qualifying = []
                            for comment in raw_comments:
                                if len(qualifying) >= MAX_QUALIFYING_COMMENTS:
                                    break

                                comment_text = comment.get("text", "")

                                # Must be substantive (≥60 chars) AND on-topic
                                if len(comment_text) < 60:
                                    continue
                                if not matches_topic(comment_text, [matched]):
                                    continue

                                likes = comment.get("like_count", 0) or 0
                                timestamp = comment.get("timestamp")
                                try:
                                    comment_time = datetime.fromtimestamp(timestamp, tz=timezone.utc) if timestamp else created_at
                                except Exception:
                                    comment_time = created_at

                                qualifying.append(build_post(
                                    platform        = "youtube",
                                    platform_id     = f"comment_{video_id}_{comment.get('id', '')}",
                                    author          = comment.get("author", "anonymous"),
                                    content         = comment_text,
                                    topic           = matched,
                                    source_category = "social",
                                    source_name     = "youtube",
                                    created_at      = comment_time,
                                    engagement      = likes,
                                    metadata        = {"type": "comment", "video_id": video_id, "video_title": title},
                                ))

                            posts.extend(qualifying)
                            print(f"  YouTube {video_id}: {len(qualifying)} qualifying comments (from {len(raw_comments)} raw)")

                    except Exception as e:
                        print(f"  Comment extraction failed for {video_id}: {e}")

            except Exception as e:
                print(f"  YouTube channel error ({channel_url}): {e}")

    return posts

This adapter fetches public posts from Mastodon across multiple instances (mastodon.social, fosstodon.org, infosec.exchange). It uses the public `/api/v2/search` endpoint — no authentication required — to search by keyword, strips HTML from post content, and formats results into the unified post schema under `source_category = "forum"`.

In [ ]:
# Mastodon adapter

def fetch_mastodon(topics, instances=None, limit_per_instance=40):
    """
    NO API KEY NEEDED — uses Mastodon's public /api/v2/search endpoint.

    Queried instances:
      - mastodon.social   (largest general instance)
      - fosstodon.org     (tech / open-source community)
      - infosec.exchange  (privacy / security community)

    Each instance is searched independently per topic keyword.
    platform_id = "{instance_domain}_{status_id}" to avoid collisions
    across instances.

    source_category = "forum"
    source_name     = instance domain (e.g. "mastodon.social")
    """
    if instances is None:
        instances = MASTODON_INSTANCES

    posts = []

    # Pick the 3 most specific keywords per topic to keep request volume low
    TOPIC_SEARCH_TERMS = {
        topic: keywords[:3]
        for topic, keywords in topics.items()
    }

    for topic, search_terms in TOPIC_SEARCH_TERMS.items():
        for term in search_terms:
            for instance in instances:
                try:
                    url = f"{instance}/api/v2/search"
                    params = {
                        "q":       term,
                        "type":    "statuses",
                        "limit":   limit_per_instance,
                        "resolve": "false",
                    }
                    # Graceful fallback if cell-6 wasn't re-run after a runtime restart
                    token = globals().get("MASTODON_ACCESS_TOKEN")
                    headers = {}
                    if token:
                        headers["Authorization"] = f"Bearer {token}"

                    response = requests.get(url, params=params, headers=headers, timeout=10)
                    if response.status_code != 200:
                        print(f"  Mastodon {instance} returned {response.status_code} for '{term}'")
                        continue

                    statuses = response.json().get("statuses", [])

                    for status in statuses:
                        content_text = clean_html(status.get("content", ""))
                        if len(content_text) < 20:
                            continue

                        matched = matches_topic(content_text, [topic])
                        if not matched:
                            continue

                        created_at_str = status.get("created_at", "")
                        try:
                            created_at = datetime.fromisoformat(created_at_str.replace("Z", "+00:00"))
                        except Exception:
                            created_at = datetime.now(timezone.utc)

                        favourites = status.get("favourites_count", 0) or 0
                        reblogs    = status.get("reblogs_count",    0) or 0
                        replies    = status.get("replies_count",    0) or 0

                        instance_domain = instance.replace("https://", "")

                        post = build_post(
                            platform        = "mastodon",
                            platform_id     = f"{instance_domain}_{status.get('id')}",
                            author          = status.get("account", {}).get("acct", "anonymous"),
                            content         = truncate_content(content_text),
                            topic           = matched,
                            source_category = "forum",
                            source_name     = instance_domain,
                            created_at      = created_at,
                            engagement      = favourites + reblogs + replies,
                            metadata        = {
                                "instance":         instance,
                                "url":              status.get("url"),
                                "favourites_count": favourites,
                                "reblogs_count":    reblogs,
                                "replies_count":    replies,
                                "language":         status.get("language"),
                            },
                        )
                        posts.append(post)

                except Exception as e:
                    print(f"  Mastodon error ({instance}, '{term}'): {e}")

    return posts

This adapter uses the Guardian API to fetch recent news articles related to tracked topics. It constructs API requests, parses the search results, extracts article content, author, and publication date, then formats the data, noting that engagement metrics are not available.

In [8]:
# Guardian adapter

def fetch_guardian(api_key, topics, limit=20):
    """
    NEEDS FREE API KEY
    Register: https://open-platform.theguardian.com/access/

    Try in your browser (add your key):
    https://content.guardianapis.com/search?q=ai+regulation&api-key=YOUR_KEY&show-fields=bodyText,headline&page-size=5

    Endpoint: https://content.guardianapis.com/search

    Questions:
    - What query params does it accept?
      (q, section, from-date, page-size, show-fields, order-by, api-key)
    - "show-fields" — which fields do you need?
      (bodyText, headline, byline, thumbnail, trailText)
    - "order-by" can be "newest" — use it
    - "from-date" — format is YYYY-MM-DD
    - How do you limit to recent articles only?
    - Rate limit: 12 calls/sec — generous but respect it
    - Guardian content is NEWS — different from forum opinions
    - For opinion tracking, "trailText" (summary) might be
      more useful than full bodyText

    source_category = "news"
    source_name = "guardian"
    """
    posts = []

    GUARDIAN_URL = "https://content.guardianapis.com/search"
    yesterday = (datetime.now(timezone.utc) - timedelta(days=1)).strftime('%Y-%m-%d')

    for topic in topics:
        params = {
            "q": topic, # Search query
            "api-key": api_key,
            "page-size": limit,
            "order-by": "newest",
            "show-fields": "bodyText,headline,byline,webUrl,trailText", # Request all fields to potentially use
            "from-date": yesterday,
        }

        data = safe_request(GUARDIAN_URL, params=params)
        if not data or "response" not in data or "results" not in data["response"]:
            continue

        for article in data["response"]["results"]:
            # Use headline + trailText as content for topic matching and post content
            content_to_match = f"{article.get('webTitle', '')} {article.get('fields', {}).get('trailText', '')} {article.get('fields', {}).get('bodyText', '')}"
            matched_topic = matches_topic(content_to_match, [topic])

            if matched_topic:
                published_at_str = article.get('webPublicationDate')
                created_at = datetime.fromisoformat(published_at_str.replace('Z', '+00:00')) if published_at_str else datetime.now(timezone.utc)

                # Use webUrl as platform_id for uniqueness
                platform_id = article.get('webUrl')
                # Author byline can be in 'byline' field within 'fields'
                author = article.get('fields', {}).get('byline', 'anonymous')
               # Combine headline + trailText + bodyText snippet
                headline = article.get('webTitle', '')
                trail = article.get('fields', {}).get('trailText', '')
                body = article.get('fields', {}).get('bodyText', '')

                if trail and "live blog" not in trail.lower() and "closed" not in trail.lower():
                    content = truncate_content(f"{headline}. {trail}")
                elif body:
                    content = truncate_content(f"{headline}. {body}")
                else:
                    content = headline

                post = build_post(
                    platform="guardian",
                    platform_id=platform_id,
                    author=author,
                    content=content,
                    topic=matched_topic,
                    source_category="news",
                    source_name="guardian",
                    created_at=created_at,
                    engagement=0, # Guardian API typically doesn't provide engagement metrics
                    metadata=article # Store full article as raw_metadata
                )
                posts.append(post)

    return posts

This adapter processes a dictionary of RSS feeds, parsing entries from each URL. It extracts titles, summaries, and full content (if available) for topic matching, cleans HTML, converts publication dates, and then builds standardized post objects.

In [9]:
# RSS feeds adapter

import feedparser
import time
from datetime import datetime, timezone, timedelta
import calendar

def fetch_rss_feeds(feeds_dict, topics, limit_per_feed=20):
    """
    NO API KEY NEEDED

    feedparser library handles all RSS/Atom formats

    Try it:
    >>> import feedparser
    >>> feed = feedparser.parse("http://feeds.bbci.co.uk/news/rss.xml")
    >>> feed.entries[0].keys()

    Questions:
    - What fields does a feed entry have?
      (title, summary, link, published, published_parsed, author)
    - "summary" often contains HTML — clean it
    - "published_parsed" is a time.struct_time — how to convert to datetime?
    - Not all feeds have "author" — handle gracefully
    - Some feeds have "content" (full text), some only "summary" (snippet)
    - How do you avoid re-fetching articles you already have?
    - What's your platform_id? (hint: the article URL is unique)

    Bonus RSS feeds to research:
    - AP News RSS
    - TechCrunch RSS
    - Ars Technica RSS
    - The Verge RSS

    source_category = "news"
    source_name = feed name from feeds_dict (e.g., "bbc", "reuters")
    """
    posts = []

    for feed_name, feed_url in feeds_dict.items():
        try:
            feed = feedparser.parse(feed_url)

            # 1. Iterate through feed.entries[:limit_per_feed]
            for entry in feed.entries[:limit_per_feed]:
                # 2. Extract title + summary, clean HTML
                title = entry.get('title', '')
                summary = clean_html(entry.get('summary', ''))
                # Some RSS feeds might have 'content' which is richer
                content_full = ''
                if 'content' in entry and len(entry['content']) > 0: # Entry 'content' is a list of dicts
                    # Take the first content element's value, if it exists
                    content_full = clean_html(entry['content'][0].get('value', ''))

                # Combine title, summary, and full content for topic matching
                combined_text = f"{title} {summary} {content_full}"

                # 3. Check topic match against title + summary
                topic = matches_topic(combined_text, topics)

                if topic:
                    # 4. Parse published date
                    #    entry.published_parsed → time.struct_time
                    #    Convert using calendar.timegm() and datetime.fromtimestamp()
                    created_at = None
                    if entry.get('published_parsed'):
                        created_at = datetime.fromtimestamp(calendar.timegm(entry.published_parsed), tz=timezone.utc)
                    else:
                        # Fallback to current time if no published date
                        created_at = datetime.now(timezone.utc)

                    # 5. build_post() with:
                    #    platform = "rss"
                    #    platform_id = entry.link (the URL) - ensures uniqueness
                    #    source_name = feed_name
                    #    source_category = "news"
                    #    engagement = 0 (RSS doesn't give this)

                    post = build_post(
                        platform="rss",
                        platform_id=entry.get('link'),
                        author=entry.get('author', 'anonymous'),
                        content=truncate_content(f"{title} {summary}" if not content_full else content_full),
                        topic=topic,
                        source_category="news",
                        source_name=feed_name,
                        created_at=created_at,
                        engagement=0,
                        metadata=entry # Store full entry as raw_metadata
                    )
                    posts.append(post)

        except Exception as e:
            print(f"RSS error for {feed_name}: {e}")

    return posts

This function implements a three-level deduplication strategy. It first checks for exact platform ID matches, then for URL matches within content, and finally uses TF-IDF cosine similarity to detect near-duplicate text across different sources, ensuring unique posts are stored.

In [10]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def deduplicate(new_posts, existing_posts):
    """
    6 sources means HIGH chance of duplicates

    Same story will appear on: Guardian + BBC RSS + Bluesky shares

    Three levels of dedup:

    Level 1: Exact platform_id match
    - Already handled by Supabase UNIQUE constraint
    - But you should skip before even trying to insert (saves API calls)

    Level 2: URL match
    - Guardian article URL == RSS article URL == URL shared on Bluesky
    - Extract URLs from content using regex
    - Questions: what regex pattern catches URLs?

    Level 3: Text similarity (cross-platform)
    - BBC writes "AI bill passes parliament"
    - Guardian writes "Parliament approves artificial intelligence legislation"
    - Same story, different words
    - TF-IDF cosine similarity > 0.85 = likely duplicate

    Think about:
    - Which post do you KEEP when you find a dupe? The one with higher engagement? The earliest one?
    - Performance: comparing every new post against every existing
      post is O(n*m). At what point does this become a problem?
    """
    unique_posts = []
    if not existing_posts:
        return new_posts # No existing posts, return all new posts

    seen_platform_ids = {p['platform_id'] for p in existing_posts}
    seen_urls = set()

    # Pre-populate seen_urls from existing posts
    url_pattern = re.compile(r'https?://(?:[-\w.]|(?:%[\da-fA-F]{2}))+')
    for post in existing_posts:
        urls_in_content = url_pattern.findall(post['content'])
        for url in urls_in_content:
            seen_urls.add(url)

    # Separate new posts into those with enough content for similarity check
    posts_for_similarity = []
    for post in new_posts:
        if post['platform_id'] in seen_platform_ids:
            continue # Skip Level 1 duplicates

        urls_in_content = url_pattern.findall(post['content'])
        if any(url in seen_urls for url in urls_in_content):
            continue # Skip Level 2 duplicates

        # If post has sufficient content, add to list for similarity check
        if len(post['content']) > 50: # Arbitrary threshold for meaningful text
            posts_for_similarity.append(post)
        else:
            unique_posts.append(post) # Add short posts directly if no L1/L2 dupe
            seen_platform_ids.add(post['platform_id'])
            for url in urls_in_content: seen_urls.add(url)

    if not posts_for_similarity:
        return unique_posts # No posts left for similarity check

    existing_long = [p['content'] for p in existing_posts if len(p['content']) > 50]

    if not existing_long:
        return new_posts # No existing posts with enough content for similarity check

    # Prepare texts for TF-IDF
    all_texts = existing_long + [p['content'] for p in posts_for_similarity]

    # Handle case where all_texts might be empty (e.g., all existing posts too short)
    if not all_texts:
        return new_posts # If no basis for comparison, consider all new posts unique

    vectorizer = TfidfVectorizer().fit(all_texts)
    existing_vectors = vectorizer.transform([p['content'] for p in existing_posts if len(p['content']) > 50])

    for new_post in posts_for_similarity:
        is_duplicate = False
        new_post_vector = vectorizer.transform([new_post['content']])

        if existing_vectors.shape[0] > 0: # Ensure there are existing vectors to compare against
            similarities = cosine_similarity(new_post_vector, existing_vectors)
            if similarities.max() > 0.85: # Level 3: Text similarity threshold
                is_duplicate = True

        if not is_duplicate:
            unique_posts.append(new_post)
            seen_platform_ids.add(new_post['platform_id'])
            for url in url_pattern.findall(new_post['content']): seen_urls.add(url)

    return unique_posts

This cell provides core Supabase interaction functions. `save_posts` handles batch upserting new posts, `get_recent_posts` retrieves data for deduplication, and `get_health_stats` offers insights into pipeline performance and database content.

In [11]:
def save_posts(client, posts):
    if not posts:
        return 0

    # Remove duplicates within the batch
    seen = set()
    unique = []
    for post in posts:
        key = (post["platform"], post["platform_id"])
        if key not in seen:
            seen.add(key)
            unique.append(post)

    try:
        response = client.table("posts").upsert(
            unique, on_conflict="platform,platform_id"
        ).execute()
        print(f"Successfully upserted {len(unique)} posts.")
        return len(unique)
    except Exception as e:
        print(f"Error saving posts to Supabase: {e}")
        return 0

def get_recent_posts(client, hours=6):
    cutoff_time = datetime.now(timezone.utc) - timedelta(hours=hours)
    try:
        response = client.table("posts").select("platform,platform_id,content").gte("ingested_at", cutoff_time.isoformat()).execute()
        if response.data:
            return response.data
        return []
    except Exception as e:
        print(f"Error fetching recent posts from Supabase: {e}")
        return []

def get_health_stats(client):
    try:
        print("\n--- Supabase Health Stats ---")

        # Total DB rows
        total_response = client.table("posts").select("*", count="exact").execute()
        total_rows = total_response.count
        print(f"Total posts in DB: {total_rows}")

        # Posts in the last hour
        one_hour_ago = (datetime.now(timezone.utc) - timedelta(hours=1)).isoformat()
        recent_response = client.table("posts").select("source_name,source_category,topic").gte("ingested_at", one_hour_ago).execute()
        recent_posts = recent_response.data if recent_response.data else []

        if recent_posts:
            print("\nPosts in last hour:")

            source_name_counts = Counter(p['source_name'] for p in recent_posts)
            for name, count in source_name_counts.most_common():
                print(f"  Source '{name}': {count}")

            source_category_counts = Counter(p['source_category'] for p in recent_posts)
            for category, count in source_category_counts.most_common():
                print(f"  Category '{category}': {count}")

            topic_counts = Counter(p['topic'] for p in recent_posts)
            if topic_counts:
                print("\nTopics in last hour:")
                for topic, count in topic_counts.most_common():
                    print(f"  '{topic}': {count}")
        else:
            print("No posts ingested in the last hour.")

        print("---------------------------")

    except Exception as e:
        print(f"Error getting health stats: {e}")

from collections import Counter

In [12]:
# Test 1: Can yt-dlp find videos?
#result = subprocess.run(
 #   [
 #       "yt-dlp",
 #       "--flat-playlist",
 #       "--playlist-end", "2",
 #       "--print", "%(id)s|||%(title)s",
 #       "https://www.youtube.com/@MatthewBerman/videos"
 #   ],
 #   capture_output=True, text=True, timeout=30
#)
#print("STDOUT:", result.stdout[:500])
#print("STDERR:", result.stderr[:500])

In [13]:
#def check_channel(url):
#    result = subprocess.run(
#        ["yt-dlp", "--flat-playlist", "--playlist-end", "3", "--print", "%(title)s", f"{url}/videos"],
#        capture_output=True, text=True, timeout=30
#    )
#    print(f"\n{url}")
#    print(result.stdout[:300] if result.stdout else f"ERROR: {result.stderr[:200]}")

#check_channel("https://www.youtube.com/@matthew_berman")
#check_channel("https://www.youtube.com/@aiaborjmustafa")
#check_channel("https://www.youtube.com/@mkbhd")

This is the main pipeline function, orchestrating the entire data ingestion process. It iteratively calls each source adapter, collects all fetched posts, performs deduplication, stores the unique posts in Supabase, and provides health statistics for monitoring.

In [ ]:
import time
from datetime import datetime, timezone, timedelta

def run_pipeline(cycles=10, interval_seconds=300):
    """
    Runs all 5 adapters → dedup → store → report

    Sources: YouTube, Mastodon, Bluesky, Guardian, RSS
    """

    total_ingested = 0

    for cycle in range(cycles):
        print(f"\n{'='*60}")
        print(f"  CYCLE {cycle+1}/{cycles} — {datetime.now(timezone.utc).strftime('%H:%M:%S UTC')}")
        print(f"{'='*60}")

        all_posts = []
        source_counts = {}

        # ── SOCIAL ──

        # YouTube Comments
        try:
            yt = fetch_youtube(TRACKED_TOPICS)
            source_counts["youtube"] = len(yt)
            all_posts.extend(yt)
        except Exception as e:
            source_counts["youtube"] = f"ERROR: {e}"

        # Bluesky
        try:
            bsky = fetch_bluesky(BLUESKY_HANDLE, BLUESKY_APP_PASSWORD, TRACKED_TOPICS)
            source_counts["bluesky"] = len(bsky)
            all_posts.extend(bsky)
        except Exception as e:
            source_counts["bluesky"] = f"ERROR: {e}"

        # ── FORUMS ──

        # Mastodon
        try:
            mastodon = fetch_mastodon(TRACKED_TOPICS)
            source_counts["mastodon"] = len(mastodon)
            all_posts.extend(mastodon)
        except Exception as e:
            source_counts["mastodon"] = f"ERROR: {e}"

        # ── NEWS ──

        # Guardian
        try:
            guardian = fetch_guardian(GUARDIAN_API_KEY, TRACKED_TOPICS)
            source_counts["guardian"] = len(guardian)
            all_posts.extend(guardian)
        except Exception as e:
            source_counts["guardian"] = f"ERROR: {e}"

        # RSS Feeds
        try:
            rss = fetch_rss_feeds(RSS_FEEDS, TRACKED_TOPICS)
            source_counts["rss"] = len(rss)
            all_posts.extend(rss)
        except Exception as e:
            source_counts["rss"] = f"ERROR: {e}"

        # ── Report collection ──
        print("\n  Source breakdown:")
        for source, count in source_counts.items():
            status = f"  ✓ {source}: {count} posts" if isinstance(count, int) else f"  ✗ {source}: {count}"
            print(f"    {status}")

        print(f"\n  Total collected: {len(all_posts)}")

        # ── Deduplicate ──
        recent = get_recent_posts(supabase)
        unique = deduplicate(all_posts, recent)
        print(f"  After dedup: {len(unique)} posts")
        print(f"  Duplicates removed: {len(all_posts) - len(unique)}")

        # ── Store ──
        saved = save_posts(supabase, unique)
        total_ingested += saved
        print(f"  Saved to Supabase: {saved}")
        print(f"  Running total: {total_ingested} posts")

        # ── Health Stats ──
        get_health_stats(supabase)

        # ── Sleep ──
        if cycle < cycles - 1:
            print(f"\n  Next cycle in {interval_seconds}s...")
            time.sleep(interval_seconds)

    print(f"\n{'='*60}")
    print(f"  PIPELINE COMPLETE — {total_ingested} total posts ingested")
    print(f"{'='*60}")

# Run it!
run_pipeline(cycles=6, interval_seconds=300)

This function provides a verification checklist for Phase 1 of the project. It queries Supabase to report statistics such as post counts per source, category, and topic, checks for null content, and displays the timestamp range and total row count.

In [16]:
def verify_phase1():
    print("\n--- Phase 1 Verification ---")
    try:
        # Fetch all posts (select only needed columns)
        all_posts = supabase.table("posts").select("source_name,source_category,topic,content,created_at,engagement").execute()
        posts = all_posts.data if all_posts.data else []

        if not posts:
            print("No posts in database yet.")
            return

        # 1. Count per source_name
        print("\n1. Posts per Source:")
        source_counts = Counter(p['source_name'] for p in posts)
        for name, count in source_counts.most_common():
            print(f"  {name}: {count} posts")

        # 2. Count per source_category
        print("\n2. Posts per Category:")
        category_counts = Counter(p['source_category'] for p in posts)
        for category, count in category_counts.most_common():
            print(f"  {category}: {count} posts")

        # 3. Count per topic
        print("\n3. Posts per Topic:")
        topic_counts = Counter(p['topic'] for p in posts)
        for topic, count in topic_counts.most_common():
            print(f"  {topic}: {count} posts")

        # 4. Check for nulls in content
        null_count = sum(1 for p in posts if not p.get('content'))
        print(f"\n4. Posts with Null Content: {null_count}")

        # 5. Oldest and newest timestamps
        timestamps = sorted([p['created_at'] for p in posts if p.get('created_at')])
        print(f"\n5. Oldest post: {timestamps[0] if timestamps else 'N/A'}")
        print(f"   Newest post: {timestamps[-1] if timestamps else 'N/A'}")

        # 6. Total row count
        print(f"\n6. Total posts in DB: {len(posts)}")

    except Exception as e:
        print(f"Error during Phase 1 Verification: {e}")
    print("----------------------------")

verify_phase1()


--- Phase 1 Verification ---

1. Posts per Source:
  youtube: 532 posts
  bluesky: 304 posts
  guardian: 79 posts
  devto: 15 posts
  techcrunch: 15 posts
  al_jazeera: 14 posts
  ars_technica: 12 posts
  bbc: 9 posts
  south_china_morning_post: 6 posts
  cnn_technology: 5 posts
  nieman_lab: 3 posts
  npr: 3 posts
  france24: 2 posts
  science_daily: 1 posts

2. Posts per Category:
  social: 836 posts
  news: 149 posts
  forum: 15 posts

3. Posts per Topic:
  tech news: 209 posts
  climate policy: 188 posts
  artificial intelligence: 147 posts
  world news: 144 posts
  us politics: 119 posts
  hollywood: 87 posts
  tech layoffs: 63 posts
  data privacy: 29 posts
  business: 14 posts

4. Posts with Null Content: 15

5. Oldest post: 2026-02-02T00:00:00+00:00
   Newest post: 2026-05-01T20:38:51.045966+00:00

6. Total posts in DB: 1000
----------------------------


In [ ]:
def cleanup_youtube_comments(client, dry_run=True):
    """
    Deletes YouTube comment posts from Supabase that don't match their
    assigned topic. Video posts (title+description) are left untouched.

    Parameters
    ----------
    client   : Supabase client
    dry_run  : If True, only prints what would be deleted. Set to False to commit.
    """
    print("Fetching all YouTube posts from Supabase...")
    response = client.table("posts") \
        .select("platform_id, content, topic") \
        .eq("platform", "youtube") \
        .execute()

    all_yt = response.data or []
    comment_posts = [p for p in all_yt if p["platform_id"].startswith("comment_")]
    video_posts   = [p for p in all_yt if p["platform_id"].startswith("video_")]

    print(f"  Total YouTube rows  : {len(all_yt)}")
    print(f"  Video posts         : {len(video_posts)}  (not touched)")
    print(f"  Comment posts       : {len(comment_posts)}")

    to_delete = [
        p["platform_id"]
        for p in comment_posts
        if not matches_topic(p["content"], [p["topic"]])
    ]

    kept = len(comment_posts) - len(to_delete)
    print(f"\n  Off-topic comments to delete : {len(to_delete)}")
    print(f"  On-topic comments to keep    : {kept}")
    print(f"  Noise rate                   : {len(to_delete)/len(comment_posts)*100:.1f}%" if comment_posts else "")

    if dry_run:
        print("\n  DRY RUN — no changes made. Pass dry_run=False to commit.")
        return

    if not to_delete:
        print("\n  Nothing to delete.")
        return

    # Delete in batches of 100 (Supabase in-filter limit)
    deleted_total = 0
    for i in range(0, len(to_delete), 100):
        batch = to_delete[i:i + 100]
        client.table("posts").delete().in_("platform_id", batch).execute()
        deleted_total += len(batch)
        print(f"  Deleted batch {i//100 + 1}: {len(batch)} rows  (running total: {deleted_total})")

    print(f"\n  Done. {deleted_total} off-topic comment rows removed from Supabase.")


# ── Preview first ──
cleanup_youtube_comments(supabase, dry_run=True)

# ── Then commit ──
# cleanup_youtube_comments(supabase, dry_run=False)

### YouTube Comment Cleanup — Existing Data

The original `fetch_youtube()` stored all comments without topic filtering, so Supabase contains
noise (reaction comments, off-topic replies, "first!", etc.).

`cleanup_youtube_comments()` removes those retroactively:
- Fetches every YouTube comment row (`platform_id` starts with `comment_`)
- Runs each through `matches_topic()` against the topic already assigned to it
- Deletes rows that fail — they were misclassified noise
- Video rows (`platform_id` starts with `video_`) are untouched — they used title+description which is already clean

Run with `dry_run=True` first to see how many rows will be removed, then `dry_run=False` to commit.